## Import Libraries

In [ ]:
def try_import(name, import_stmt):
    try:
        exec(import_stmt, globals())
        print(f"{name} imported successfully")
    except Exception as e:
        print(f"Failed to import {name}: {e}")

try_import("os", "import os")
try_import("shutil", "import shutil")
try_import("zipfile", "import zipfile")
try_import("json", "import json")
try_import("numpy", "import numpy as np")
try_import("pandas", "import pandas as pd")
try_import("pathlib", "from pathlib import Path")
try_import("collections", "from collections import defaultdict, Counter")
try_import("PIL", "from PIL import Image")
try_import("matplotlib.pyplot", "import matplotlib.pyplot as plt")
try_import("matplotlib.patches", "import matplotlib.patches as patches")
try_import("sklearn.model_selection", "from sklearn.model_selection import train_test_split")
try_import("datetime", "from datetime import datetime")
try_import("warnings", "import warnings; warnings.filterwarnings('ignore')")

## Download Dataset from Google Drive

In [ ]:
os.system('pip install gdown -q')

BASE_DATA_PATH = './data'
GDRIVE_FILE_ID = '1hqnsrIbKeYHyxyKIPKoOedsLWLInKnh0'

os.makedirs(BASE_DATA_PATH, exist_ok=True)
download_path = os.path.join(BASE_DATA_PATH, 'dataset.zip')

# check if file already exists
if os.path.exists(download_path) and os.path.getsize(download_path) > 0:
    file_size_mb = os.path.getsize(download_path) / (1024 * 1024)
    print(f"Already exists in: {download_path}")
    print(f"Size: {file_size_mb:.2f} MB")
else:
    print("Downloading dataset from Google Drive...")
    print("Note: If using a different machine, update BASE_DATA_PATH and ensure you have Google Drive access")

    os.system(f'gdown {GDRIVE_FILE_ID} -O {download_path} -q')

    if os.path.exists(download_path) and os.path.getsize(download_path) > 0:
        file_size_mb = os.path.getsize(download_path) / (1024 * 1024)

        print(f"\n[SUCCESS] Dataset downloaded successfully!")
        print(f"  File: {download_path}")
        print(f"  Size: {file_size_mb:.2f} MB")
        print("\nSuccessfully downloaded dataset from Google Drive!")

    else:
        print(f"\n[FAILED] Download failed or file is empty!")
        print(f"  Check your internet connection and Google Drive access")

## Unzip Dataset and Clean Up

In [ ]:
print("\nChecking zip integrity...")

# ALWAYS define paths first (important fix)
dataset_folder = os.path.join(BASE_DATA_PATH, "Dataset")
extracted_path = BASE_DATA_PATH  # safe default so other cells never break

# check if already unzipped
if os.path.exists(dataset_folder) and len(os.listdir(dataset_folder)) > 0:
    print(f"Dataset already extracted at: {dataset_folder}")
    print("Skipping unzip step")

else:

    try:
        with zipfile.ZipFile(download_path, 'r') as zip_ref:
            bad_file = zip_ref.testzip()
            if bad_file is None:
                print("Zip file is valid")
            else:
                print(f"Corrupted file found inside zip: {bad_file}")
                raise Exception("Zip file is corrupted")

    except Exception as e:
        print(f"[FAILED] Zip validation failed: {e}")
        raise


    print("\nExtracting and cleaning up dataset...")

    # remove old Dataset folder if it exists (prevents nesting issues)
    if os.path.exists(dataset_folder):
        shutil.rmtree(dataset_folder)
        print(f"Removed existing folder: {dataset_folder}")

    try:
        with zipfile.ZipFile(download_path, 'r') as zip_ref:
            zip_ref.extractall(extracted_path)

        if os.path.exists(dataset_folder) and len(os.listdir(dataset_folder)) > 0:
            print(f"[SUCCESS] Dataset extracted successfully!")
            print(f"  Path: {dataset_folder}")
        else:
            print(f"[FAILED] Extraction failed or Dataset folder missing!")
            raise Exception("Extraction did not produce expected Dataset folder")

    except Exception as e:
        print(f"[FAILED] Error during extraction: {e}")
        raise


    removed_macosx_dirs = 0
    removed_ds_store_files = 0

    for root, dirs, files in os.walk(extracted_path):
        if '__MACOSX' in dirs:
            macosx_path = os.path.join(root, '__MACOSX')
            shutil.rmtree(macosx_path)
            removed_macosx_dirs += 1
            print(f"  Removed: {macosx_path}")
        
        for file in files:
            if file == '.DS_Store':
                ds_store_path = os.path.join(root, file)
                os.remove(ds_store_path)
                removed_ds_store_files += 1
                print(f"  Removed: {ds_store_path}")


    # remove any leftover .part files
    removed_part_files = 0

    for file in os.listdir(BASE_DATA_PATH):
        if file.endswith(".part"):
            part_path = os.path.join(BASE_DATA_PATH, file)
            os.remove(part_path)
            removed_part_files += 1
            print(f"  Removed partial file: {part_path}")


    print(f"\n[SUCCESS] Cleanup completed!")
    print(f"  __MACOSX directories removed: {removed_macosx_dirs}")
    print(f"  .DS_Store files removed: {removed_ds_store_files}")
    print(f"  .part files removed: {removed_part_files}")

## Discover Dataset Structure

In [ ]:
print("\nDiscovering dataset structure...")

images_dir = None
labels_dir = None
coco_json_file = None
dataset_format = None

# flags to support mixed-format datasets (COCO + YOLO txt)
yolo_labels_available = False
coco_annotations_available = False

for root, dirs, files in os.walk(extracted_path):
    if images_dir and labels_dir and coco_json_file:
        break

    if not images_dir:
        for d in dirs:
            if d.lower() in ['images', 'imgs', 'img', 'pictures']:
                images_dir = os.path.join(root, d)
                break

    if not labels_dir:
        for d in dirs:
            if d.lower() in ['labels', 'annotations', 'annot']:
                labels_dir = os.path.join(root, d)
                break

    for file in files:
        if file.endswith('.json') and 'coco' in file.lower():
            coco_json_file = os.path.join(root, file)
            break

if images_dir and labels_dir:
    print(f"[SUCCESS] Dataset structure discovered!")
    print(f"  Images directory: {images_dir}")
    print(f"  Labels directory: {labels_dir}")

    image_files = [f for f in os.listdir(images_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    label_files = [f for f in os.listdir(labels_dir) if f.lower().endswith(('.txt', '.xml'))]
    txt_label_files = [f for f in os.listdir(labels_dir) if f.lower().endswith('.txt')]

    yolo_labels_available = len(txt_label_files) > 0
    coco_annotations_available = coco_json_file is not None

    print(f"\nInitial counts:")
    print(f"  Image files: {len(image_files)}")
    print(f"  Label files: {len(label_files)}")
    print(f"  YOLO txt labels: {len(txt_label_files)}")

    if coco_annotations_available:
        dataset_format = 'COCO'
        print(f"  Dataset format detected: COCO")
        print(f"[SUCCESS] COCO annotations found!")
    else:
        dataset_format = 'YOLO'
        print(f"  Dataset format detected: YOLO")
        print(f"[SUCCESS] YOLO format detected!")

    if coco_annotations_available and yolo_labels_available:
        print("[SUCCESS] Mixed dataset detected: COCO annotations + YOLO txt labels")
        print("  COCO will be used as primary split reference")
        print("  YOLO txt labels will also be copied to split labels folders")

else:
    print(f"[FAILED] Could not find images/labels directories!")
    if not images_dir:
        print(f"  Missing: images directory")
    if not labels_dir:
        print(f"  Missing: labels directory")

## Check if all images and labels are not corrupt

In [ ]:
print("\nRunning full dataset integrity + consistency check...")

bad_images = []
bad_labels = []
bad_coco = []

valid_images = 0
valid_labels = 0

# -------------------------
# 1. IMAGE CHECK
# -------------------------
image_files = [
    f for f in os.listdir(images_dir)
    if f.lower().endswith(('.jpg', '.jpeg', '.png'))
]

for file in image_files:
    img_path = os.path.join(images_dir, file)

    try:
        with Image.open(img_path) as img:
            img.verify()
        valid_images += 1

    except Exception:
        bad_images.append(img_path)

# -------------------------
# 2. LABEL CHECK (YOLO .txt)
# -------------------------
label_files = [
    f for f in os.listdir(labels_dir)
    if f.endswith('.txt')
]

for file in label_files:
    label_path = os.path.join(labels_dir, file)

    try:
        with open(label_path, "r") as f:
            lines = f.readlines()

        # validate YOLO format: class x_center y_center width height
        for line in lines:
            parts = line.strip().split()

            if len(parts) < 5:
                raise ValueError("Invalid YOLO format")

            # check numeric values
            [float(x) for x in parts[1:]]

        valid_labels += 1

    except Exception:
        bad_labels.append(label_path)

# -------------------------
# 3. COCO JSON CHECK
# -------------------------
coco_path = os.path.join(BASE_DATA_PATH, "Dataset", "coco.json")

if os.path.exists(coco_path):
    try:
        with open(coco_path, "r") as f:
            coco_data = json.load(f)

        # basic structure check
        if "images" not in coco_data or "annotations" not in coco_data:
            raise ValueError("Missing COCO keys")

        coco_image_files = {img["file_name"] for img in coco_data["images"]}

        # check mismatches
        dataset_images = set(image_files)

        missing_in_images = coco_image_files - dataset_images
        missing_in_coco = dataset_images - coco_image_files

        if missing_in_images or missing_in_coco:
            bad_coco.append({
                "missing_in_dataset": list(missing_in_images),
                "missing_in_coco": list(missing_in_coco)
            })

    except Exception as e:
        bad_coco.append(str(e))
else:
    bad_coco.append("coco.json not found")

# -------------------------
# RESULTS
# -------------------------
print("\nIntegrity Check Complete")

print(f"\nImages:")
print(f"  Total scanned: {len(image_files)}")
print(f"  Valid: {valid_images}")
print(f"  Corrupted: {len(bad_images)}")

print(f"\nLabels (YOLO):")
print(f"  Total scanned: {len(label_files)}")
print(f"  Valid: {valid_labels}")
print(f"  Corrupted: {len(bad_labels)}")

print(f"\nCOCO JSON:")
print(f"  Issues found: {len(bad_coco)}")

# -------------------------
# FULL REPORTS
# -------------------------
if bad_images:
    print("\nCorrupted images:")
    for p in bad_images:
        print(" ", p)

if bad_labels:
    print("\nCorrupted labels:")
    for p in bad_labels:
        print(" ", p)

if bad_coco:
    print("\nCOCO issues:")
    print(bad_coco)

print("\nTotal dataset health check complete.")

## Validate Image-Label Correspondence

In [ ]:
print("\nValidating image-label correspondence...")

def get_filename_without_ext(filename):
    return os.path.splitext(filename)[0]

valid_pairs = []
orphaned_images = []
orphaned_labels = []

image_files = [f for f in os.listdir(images_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
label_files = [f for f in os.listdir(labels_dir) if f.lower().endswith(('.txt', '.xml'))]

image_names_set = {get_filename_without_ext(f) for f in image_files}
label_names_set = {get_filename_without_ext(f) for f in label_files}

for img_name in image_names_set:
    if img_name in label_names_set:
        valid_pairs.append(img_name)
    else:
        orphaned_images.append(img_name)

for label_name in label_names_set:
    if label_name not in image_names_set:
        orphaned_labels.append(label_name)

print(f"Validation Results:")
print(f"  Valid image-label pairs: {len(valid_pairs)}")
print(f"  Orphaned images (no labels): {len(orphaned_images)}")
print(f"  Orphaned labels (no images): {len(orphaned_labels)}")

if len(orphaned_images) == 0 and len(orphaned_labels) == 0:
    print(f"\n[SUCCESS] Perfect alignment! All images have matching labels and vice versa!")
else:
    print(f"\n[WARNING] Mismatches found - will be removed in next step")
    if orphaned_images:
        print(f"  Orphaned images: {orphaned_images[:5]}{'...' if len(orphaned_images) > 5 else ''}")
    if orphaned_labels:
        print(f"  Orphaned labels: {orphaned_labels[:5]}{'...' if len(orphaned_labels) > 5 else ''}")

## Remove Mismatched Files

In [ ]:
print("\nRemoving mismatched files...")

removed_orphaned_images = 0
removed_orphaned_labels = 0

for img_name in orphaned_images:
    for img_file in image_files:
        if get_filename_without_ext(img_file) == img_name:
            img_path = os.path.join(images_dir, img_file)
            os.remove(img_path)
            removed_orphaned_images += 1
            print(f"  Removed orphaned image: {img_file}")
            break

for label_name in orphaned_labels:
    for label_file in label_files:
        if get_filename_without_ext(label_file) == label_name:
            label_path = os.path.join(labels_dir, label_file)
            os.remove(label_path)
            removed_orphaned_labels += 1
            print(f"  Removed orphaned label: {label_file}")
            break

print(f"\nRemoval Summary:")
print(f"  Orphaned images removed: {removed_orphaned_images}")
print(f"  Orphaned labels removed: {removed_orphaned_labels}")
print(f"  Clean dataset pairs remaining: {len(valid_pairs)}")

if removed_orphaned_images == 0 and removed_orphaned_labels == 0:
    print(f"\n[SUCCESS] No mismatches to remove - dataset is clean!")
else:
    print(f"\n[SUCCESS] Removed {removed_orphaned_images + removed_orphaned_labels} mismatched files!")

## Extract Class Information

In [ ]:
print("\nExtracting class information...")

class_mapping = {}
class_instance_counts = defaultdict(int)
all_class_ids = set()

if dataset_format == 'YOLO':
    print("Processing YOLO format labels...")
    updated_labels_dir = labels_dir
    label_files = [f for f in os.listdir(updated_labels_dir) if f.endswith('.txt')]
    
    for label_file in label_files:
        label_path = os.path.join(updated_labels_dir, label_file)
        with open(label_path, 'r') as f:
            lines = f.readlines()
            for line in lines:
                if line.strip():
                    parts = line.strip().split()
                    class_id = int(parts[0])
                    all_class_ids.add(class_id)
                    class_instance_counts[class_id] += 1
    
    num_classes = len(all_class_ids)
    for i in range(num_classes):
        class_mapping[i] = f'class_{i}'
    
    print(f"[SUCCESS] Found {num_classes} classes in YOLO format")

elif dataset_format == 'COCO' and coco_json_file:
    print("Processing COCO format labels...")
    try:
        with open(coco_json_file, 'r') as f:
            coco_data = json.load(f)
        
        for category in coco_data.get('categories', []):
            class_id = category['id']
            class_name = category.get('name', f'class_{class_id}')
            class_mapping[class_id] = class_name
            all_class_ids.add(class_id)
        
        for annotation in coco_data.get('annotations', []):
            class_id = annotation['category_id']
            class_instance_counts[class_id] += 1
        
        print(f"[SUCCESS] Found {len(all_class_ids)} classes in COCO format")
    except Exception as e:
        print(f"[FAILED] Error reading COCO file: {e}")
        raise
else:
    print(f"[FAILED] Could not process labels - format not detected properly")

print(f"\nClass Mapping:")
for class_id in sorted(class_mapping.keys()):
    count = class_instance_counts[class_id]
    print(f"  {class_id}: {class_mapping[class_id]} ({count} instances)")

print(f"\nTotal classes: {len(class_mapping)}")
print(f"Total annotations: {sum(class_instance_counts.values())}")

## Prepare Stratified Train/Val/Test Split

In [ ]:
print("\nPreparing data for stratified split...")

TRAIN_RATIO = 0.7
VAL_RATIO = 0.15
TEST_RATIO = 0.15

# define split output paths (IMPORTANT for skip safety)
processed_path = os.path.join(BASE_DATA_PATH, "processed_dataset")
train_path = os.path.join(processed_path, "train")
val_path = os.path.join(processed_path, "val")
test_path = os.path.join(processed_path, "test")

# check if already split
if (
    os.path.exists(train_path) and os.path.exists(val_path) and os.path.exists(test_path)
    and len(os.listdir(train_path)) > 0
    and len(os.listdir(val_path)) > 0
    and len(os.listdir(test_path)) > 0
):
    print("Dataset already split into train/val/test")
    print("Skipping stratified split step")

    # still define empty variables so later cells don’t crash
    train_images, val_images, test_images = [], [], []

else:

    image_to_class = {}

    if dataset_format == 'YOLO':
        label_files = [f for f in os.listdir(labels_dir) if f.endswith('.txt')]
        for label_file in label_files:
            img_name = get_filename_without_ext(label_file)
            label_path = os.path.join(labels_dir, label_file)

            with open(label_path, 'r') as f:
                first_line = f.readline().strip()
                if first_line:
                    class_id = int(first_line.split()[0])
                    image_to_class[img_name] = class_id
                else:
                    image_to_class[img_name] = 0

    elif dataset_format == 'COCO' and coco_json_file:
        with open(coco_json_file, 'r') as f:
            coco_data = json.load(f)

        image_id_to_name = {
            img['id']: get_filename_without_ext(img['file_name'])
            for img in coco_data.get('images', [])
        }

        for annotation in coco_data.get('annotations', []):
            image_id = annotation['image_id']
            class_id = annotation['category_id']
            if image_id in image_id_to_name:
                img_name = image_id_to_name[image_id]
                if img_name not in image_to_class:
                    image_to_class[img_name] = class_id

    image_names = list(image_to_class.keys())
    class_labels = [image_to_class[name] for name in image_names]

    print(f"Splitting {len(image_names)} images with stratified sampling...")
    print(f"  Train: {TRAIN_RATIO*100}%")
    print(f"  Validation: {VAL_RATIO*100}%")
    print(f"  Test: {TEST_RATIO*100}%")

    train_images, temp_images = train_test_split(
        image_names,
        test_size=(VAL_RATIO + TEST_RATIO),
        stratify=class_labels,
        random_state=42
    )

    temp_labels = [image_to_class[name] for name in temp_images]

    val_images, test_images = train_test_split(
        temp_images,
        test_size=TEST_RATIO / (VAL_RATIO + TEST_RATIO),
        stratify=temp_labels,
        random_state=42
    )

    print(f"\n[SUCCESS] Stratified split completed!")
    print(f"  Train: {len(train_images)} images")
    print(f"  Validation: {len(val_images)} images")
    print(f"  Test: {len(test_images)} images")
    print(f"  Class distribution preserved across all splits")

## Build Dataset Folders and Copy Images/Labels

In [ ]:
print("\nCopying files to split directories...")

processed_path = os.path.join(BASE_DATA_PATH, 'split_dataset')

splits = {
    'train': train_images,
    'val': val_images,
    'test': test_images
}

def split_ready(split_name):
    split_dir = os.path.join(processed_path, split_name)
    images_dir_split = os.path.join(split_dir, 'images')
    labels_dir_split = os.path.join(split_dir, 'labels')

    if not os.path.exists(images_dir_split):
        return False

    image_count = len([
        f for f in os.listdir(images_dir_split)
        if f.lower().endswith(('.jpg', '.jpeg', '.png'))
    ])
    if image_count == 0:
        return False

    # if YOLO txt labels exist in source, require split txt labels too
    if yolo_labels_available:
        if not os.path.exists(labels_dir_split):
            return False
        label_count = len([f for f in os.listdir(labels_dir_split) if f.endswith('.txt')])
        if label_count == 0:
            return False

    # if COCO json exists in source, require split annotation json too
    if coco_annotations_available:
        coco_output_file = os.path.join(split_dir, f'{split_name}_annotations.json')
        if not os.path.exists(coco_output_file):
            return False

    return True

all_splits_ready = all(split_ready(name) for name in ['train', 'val', 'test'])

if all_splits_ready:
    print("[SKIP] Dataset already copied into train/val/test directories")

    if coco_annotations_available:
        print("[NOTE] COCO annotations present in each split json file.")
    if yolo_labels_available:
        print("[NOTE] YOLO txt labels present in each split labels/ folder.")

else:
    if sum(len(v) for v in splits.values()) == 0:
        raise RuntimeError(
            "Split lists are empty. Run Cell 16 first (or rerun it) to populate train/val/test image lists."
        )

    os.makedirs(processed_path, exist_ok=True)
    split_image_counts = {split: 0 for split in splits}
    split_label_counts = {split: 0 for split in splits}

    for split_name, image_list in splits.items():
        split_dir = os.path.join(processed_path, split_name)
        os.makedirs(split_dir, exist_ok=True)

        split_images_dir = os.path.join(split_dir, 'images')
        split_labels_dir = os.path.join(split_dir, 'labels')
        os.makedirs(split_images_dir, exist_ok=True)
        os.makedirs(split_labels_dir, exist_ok=True)

        for img_name in image_list:
            img_file = None
            for file in os.listdir(images_dir):
                if get_filename_without_ext(file) == img_name:
                    img_file = file
                    break

            if img_file:
                src_img = os.path.join(images_dir, img_file)
                dst_img = os.path.join(split_images_dir, img_file)
                shutil.copy2(src_img, dst_img)
                split_image_counts[split_name] += 1

                # copy YOLO txt labels whenever available (even if COCO exists)
                if yolo_labels_available:
                    label_file = get_filename_without_ext(img_file) + '.txt'
                    src_label = os.path.join(labels_dir, label_file)
                    if os.path.exists(src_label):
                        dst_label = os.path.join(split_labels_dir, label_file)
                        shutil.copy2(src_label, dst_label)
                        split_label_counts[split_name] += 1

    print(f"\n[SUCCESS] Files copied to splits!")
    for split_name, count in split_image_counts.items():
        expected = len(splits[split_name])
        status = "OK" if count == expected else "MISMATCH"
        print(f"  {split_name}: {count}/{expected} images [{status}]")

    if yolo_labels_available:
        print("\nYOLO txt label copy summary:")
        for split_name, label_count in split_label_counts.items():
            print(f"  {split_name}: {label_count} txt labels copied")

    if coco_annotations_available and coco_json_file:
        print("\nCOCO annotation split summary:")
        print("[NOTE] Per-image .txt labels and COCO json can coexist for multi-model training.")

        try:
            with open(coco_json_file, 'r') as f:
                coco_data = json.load(f)

            image_id_to_name = {
                img['id']: get_filename_without_ext(img['file_name'])
                for img in coco_data.get('images', [])
            }

            for split_name, image_list in splits.items():
                split_image_names = set(image_list)

                split_images = [
                    img for img in coco_data.get('images', [])
                    if image_id_to_name.get(img['id']) in split_image_names
                ]

                split_image_ids = {img['id'] for img in split_images}

                split_annotations = [
                    ann for ann in coco_data.get('annotations', [])
                    if ann['image_id'] in split_image_ids
                ]

                split_coco = {
                    'images': split_images,
                    'annotations': split_annotations,
                    'categories': coco_data.get('categories', [])
                }

                split_dir = os.path.join(processed_path, split_name)
                coco_output_file = os.path.join(split_dir, f'{split_name}_annotations.json')

                with open(coco_output_file, 'w') as f:
                    json.dump(split_coco, f, indent=2)

                print(f"  [SUCCESS] {split_name}: {len(split_annotations)} annotations saved")

        except Exception as e:
            print(f"  [FAILED] Error splitting COCO annotations: {e}")
            raise

## Dataset Statistics

In [ ]:
print("\n" + "="*60)
print("DATASET STATISTICS")
print("="*60)

total_images = len(valid_pairs)
num_classes = len(class_mapping)

print(f"\nOverall Statistics:")
print(f"  Total images: {total_images}")
print(f"  Total classes: {num_classes}")
print(f"  Dataset format: {dataset_format}")
print(f"  Total annotations: {sum(class_instance_counts.values())}")

print(f"\nClass Distribution:")
for class_id in sorted(class_mapping.keys()):
    class_name = class_mapping[class_id]
    count = class_instance_counts[class_id]
    percentage = (count / total_images) * 100
    print(f"  {class_id}: {class_name:20s} - {count:4d} images ({percentage:5.2f}%)")

print(f"\nTrain/Val/Test Split Distribution:")
split_stats = {}
for split_name, image_list in splits.items():
    split_class_counts = defaultdict(int)
    for img_name in image_list:
        if img_name in image_to_class:
            class_id = image_to_class[img_name]
            split_class_counts[class_id] += 1
    split_stats[split_name] = split_class_counts
    print(f"\n  {split_name.upper()} ({len(image_list)} images):")
    for class_id in sorted(split_class_counts.keys()):
        class_name = class_mapping[class_id]
        count = split_class_counts[class_id]
        print(f"    {class_id}: {class_name:20s} - {count:4d}")

image_dimensions = []
print(f"\nAnalyzing image dimensions...")
sample_count = 0
for img_name in image_names[:min(100, len(image_names))]:
    img_file = None
    for file in os.listdir(images_dir):
        if get_filename_without_ext(file) == img_name:
            img_file = file
            break
    
    if img_file:
        img_path = os.path.join(images_dir, img_file)
        try:
            with Image.open(img_path) as img:
                image_dimensions.append(img.size)
                sample_count += 1
        except Exception as e:
            print(f"  Warning: Could not read {img_file}: {e}")

if image_dimensions:
    widths = [d[0] for d in image_dimensions]
    heights = [d[1] for d in image_dimensions]
    print(f"\n[SUCCESS] Image dimension analysis completed!")
    print(f"  Samples analyzed: {sample_count}/{len(image_names)}")
    print(f"  Width range: {min(widths)} - {max(widths)} pixels")
    print(f"  Height range: {min(heights)} - {max(heights)} pixels")
    print(f"  Average size: {int(np.mean(widths))}x{int(np.mean(heights))} pixels")
else:
    print(f"\n[FAILED] Could not analyze image dimensions")

## Display Sample Images From Each Class

In [ ]:
print("\nDisplaying sample images from each class...")

try:
    import random

    rng = random.Random(42)  # reproducible randomness

    fig, axes = plt.subplots(len(class_mapping), 1, figsize=(10, 3 * len(class_mapping)))

    if len(class_mapping) == 1:
        axes = [axes]

    images_displayed = 0

    shuffled_image_names = image_names.copy()
    rng.shuffle(shuffled_image_names)

    for idx, class_id in enumerate(sorted(class_mapping.keys())):
        class_name = class_mapping[class_id]

        sample_img = None

        for img_name in shuffled_image_names:
            if image_to_class.get(img_name) == class_id:

                for file in os.listdir(images_dir):
                    if get_filename_without_ext(file) == img_name:
                        img_path = os.path.join(images_dir, file)
                        sample_img = Image.open(img_path)
                        images_displayed += 1
                        break

                if sample_img:
                    break

        ax = axes[idx]

        if sample_img:
            ax.imshow(sample_img)
            ax.set_title(f"{class_name} (Class {class_id})")
            ax.axis('off')
        else:
            ax.text(0.5, 0.5, f"No image found\n{class_name}", ha='center', va='center')
            ax.axis('off')

    plt.tight_layout()

    print(f"[INFO] Displayed: {images_displayed} sample images")

    plt.show()

except Exception as e:
    print(f"[FAILED] Error displaying sample images: {e}")
    raise